# Transformer English to French

An encoder-decoder Transformer implemented in PyTorch for English-to-French translation with OPUS Books.

The architecture follows Vaswani et al., *Attention Is All You Need*: https://arxiv.org/abs/1706.03762

## 1. Configuration and reproducibility

In [ ]:
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

config = {
    'source_lang': 'en',
    'target_lang': 'fr',
    'seq_len': 128,
    'd_model': 512,
    'd_ff': 2048,
    'num_layers': 6,
    'num_heads': 8,
    'dropout': 0.1,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'seed': 42
}

random.seed(config['seed'])
torch.manual_seed(config['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 2. OPUS Books and tokenization

In [ ]:
raw_dataset = load_dataset(
    'opus_books',
    f"{config['source_lang']}-{config['target_lang']}",
    split='train'
)

def get_all_sentences(ds, lang):
    for item in ds:
        yield item['translation'][lang]

def build_tokenizer(ds, lang, path):
    path = Path(path)
    if path.exists():
        return Tokenizer.from_file(str(path))
    tokenizer = Tokenizer(WordLevel(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordLevelTrainer(
        special_tokens=['[UNK]', '[PAD]', '[SOS]', '[EOS]'],
        min_frequency=2
    )
    tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
    tokenizer.save(str(path))
    return tokenizer

tokenizer_src = build_tokenizer(raw_dataset, 'en', 'notebook_tokenizer_en.json')
tokenizer_tgt = build_tokenizer(raw_dataset, 'fr', 'notebook_tokenizer_fr.json')
tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()

## 3. Dataset representation

In [ ]:
def causal_mask(size):
    mask = torch.triu(torch.ones(1, size, size, dtype=torch.bool), diagonal=1)
    return ~mask

class BilingualDataset(Dataset):
    def __init__(self, ds, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len):
        self.ds = ds
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.seq_len = seq_len
        self.src_sos = tokenizer_src.token_to_id('[SOS]')
        self.src_eos = tokenizer_src.token_to_id('[EOS]')
        self.src_pad = tokenizer_src.token_to_id('[PAD]')
        self.tgt_sos = tokenizer_tgt.token_to_id('[SOS]')
        self.tgt_eos = tokenizer_tgt.token_to_id('[EOS]')
        self.tgt_pad = tokenizer_tgt.token_to_id('[PAD]')

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        pair = self.ds[idx]['translation']
        src_text = pair[self.src_lang]
        tgt_text = pair[self.tgt_lang]
        src_tokens = self.tokenizer_src.encode(src_text).ids
        tgt_tokens = self.tokenizer_tgt.encode(tgt_text).ids
        src_padding = self.seq_len - len(src_tokens) - 2
        tgt_padding = self.seq_len - len(tgt_tokens) - 1
        if src_padding < 0 or tgt_padding < 0:
            raise ValueError('Sentence is too long')
        encoder_input = torch.tensor([self.src_sos, *src_tokens, self.src_eos, *([self.src_pad] * src_padding)], dtype=torch.long)
        decoder_input = torch.tensor([self.tgt_sos, *tgt_tokens, *([self.tgt_pad] * tgt_padding)], dtype=torch.long)
        label = torch.tensor([*tgt_tokens, self.tgt_eos, *([self.tgt_pad] * tgt_padding)], dtype=torch.long)
        encoder_mask = (encoder_input != self.src_pad).unsqueeze(0).unsqueeze(0)
        decoder_mask = (decoder_input != self.tgt_pad).unsqueeze(0) & causal_mask(self.seq_len)
        assert encoder_input.size(0) == self.seq_len
        assert decoder_input.size(0) == self.seq_len
        assert label.size(0) == self.seq_len
        return {
            'encoder_input': encoder_input,
            'decoder_input': decoder_input,
            'encoder_mask': encoder_mask,
            'decoder_mask': decoder_mask,
            'label': label,
            'src_text': src_text,
            'tgt_text': tgt_text
        }

In [ ]:
train_size = int(0.9 * len(raw_dataset))
val_size = len(raw_dataset) - train_size
train_raw, val_raw = random_split(
    raw_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(config['seed'])
)

train_dataset = BilingualDataset(train_raw, tokenizer_src, tokenizer_tgt, 'en', 'fr', config['seq_len'])
val_dataset = BilingualDataset(val_raw, tokenizer_src, tokenizer_tgt, 'en', 'fr', config['seq_len'])
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

batch = next(iter(train_loader))
{k: (v.shape if torch.is_tensor(v) else type(v)) for k, v in batch.items()}

## 4. Embeddings, positional encoding, normalization and feed-forward network

In [ ]:
class LayerNormalization(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features))
        self.bias = nn.Parameter(torch.zeros(features))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.alpha * (x - mean) / torch.sqrt(variance + self.eps) + self.bias

class FeedForwardBlock(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

class InputEmbeddings(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, seq_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class ResidualConnection(nn.Module):
    def __init__(self, features, dropout):
        super().__init__()
        self.norm = LayerNormalization(features)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

## 5. Scaled dot-product attention and multi-head attention

In [ ]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model, h, dropout):
        super().__init__()
        if d_model % h != 0:
            raise ValueError('d_model must be divisible by h')
        self.h = h
        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.attention_scores = None

    @staticmethod
    def attention(query, key, value, mask=None, dropout=None):
        scores = query @ key.transpose(-2, -1) / math.sqrt(query.size(-1))
        if mask is not None:
            scores = scores.masked_fill(~mask.bool(), torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        if dropout is not None:
            weights = dropout(weights)
        return weights @ value, weights

    def forward(self, q, k, v, mask=None):
        batch, query_len, _ = q.shape
        key_len = k.size(1)
        value_len = v.size(1)
        query = self.w_q(q).view(batch, query_len, self.h, self.d_k).transpose(1, 2)
        key = self.w_k(k).view(batch, key_len, self.h, self.d_k).transpose(1, 2)
        value = self.w_v(v).view(batch, value_len, self.h, self.d_k).transpose(1, 2)
        x, self.attention_scores = self.attention(query, key, value, mask, self.dropout)
        x = x.transpose(1, 2).contiguous().view(batch, query_len, self.h * self.d_k)
        return self.w_o(x)

## 6. Encoder and decoder stacks

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, features, self_attention_block, feed_forward_block, dropout):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        return self.residual_connections[1](x, self.feed_forward_block)

class Encoder(nn.Module):
    def __init__(self, features, layers):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderBlock(nn.Module):
    def __init__(self, features, self_attention_block, cross_attention_block, feed_forward_block, dropout):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connections = nn.ModuleList([ResidualConnection(features, dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        x = self.residual_connections[0](x, lambda x: self.self_attention_block(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        return self.residual_connections[2](x, self.feed_forward_block)

class Decoder(nn.Module):
    def __init__(self, features, layers):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(features)

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

class ProjectionLayer(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.proj(x)

## 7. Complete Transformer model

In [ ]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, projection_layer):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        src = self.src_pos(self.src_embed(src))
        return self.encoder(src, src_mask)

    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        tgt = self.tgt_pos(self.tgt_embed(tgt))
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        return self.projection_layer(x)

def build_transformer(src_vocab_size, tgt_vocab_size, src_seq_len, tgt_seq_len, d_model=512, N=6, h=8, dropout=0.1, d_ff=2048):
    src_embed = InputEmbeddings(d_model, src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)
    encoder_blocks = [EncoderBlock(d_model, MultiHeadAttentionBlock(d_model, h, dropout), FeedForwardBlock(d_model, d_ff, dropout), dropout) for _ in range(N)]
    decoder_blocks = [DecoderBlock(d_model, MultiHeadAttentionBlock(d_model, h, dropout), MultiHeadAttentionBlock(d_model, h, dropout), FeedForwardBlock(d_model, d_ff, dropout), dropout) for _ in range(N)]
    model = Transformer(
        Encoder(d_model, nn.ModuleList(encoder_blocks)),
        Decoder(d_model, nn.ModuleList(decoder_blocks)),
        src_embed, tgt_embed, src_pos, tgt_pos,
        ProjectionLayer(d_model, tgt_vocab_size)
    )
    for parameter in model.parameters():
        if parameter.dim() > 1:
            nn.init.xavier_uniform_(parameter)
    return model

In [ ]:
model = build_transformer(
    tokenizer_src.get_vocab_size(),
    tokenizer_tgt.get_vocab_size(),
    config['seq_len'],
    config['seq_len'],
    config['d_model'],
    config['num_layers'],
    config['num_heads'],
    config['dropout'],
    config['d_ff']
).to(device)

sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

## 8. Forward pass and tensor shapes

In [ ]:
batch = next(iter(train_loader))
encoder_input = batch['encoder_input'].to(device)
decoder_input = batch['decoder_input'].to(device)
encoder_mask = batch['encoder_mask'].to(device)
decoder_mask = batch['decoder_mask'].to(device)

encoder_output = model.encode(encoder_input, encoder_mask)
decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
logits = model.project(decoder_output)
encoder_input.shape, encoder_mask.shape, decoder_input.shape, decoder_mask.shape, encoder_output.shape, decoder_output.shape, logits.shape

## 9. Training step

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], eps=1e-9)
pad_id = tokenizer_tgt.token_to_id('[PAD]')
loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id, label_smoothing=0.1).to(device)

model.train()
optimizer.zero_grad(set_to_none=True)
encoder_output = model.encode(encoder_input, encoder_mask)
decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
logits = model.project(decoder_output)
labels = batch['label'].to(device)
loss = loss_fn(logits.reshape(-1, tokenizer_tgt.get_vocab_size()), labels.reshape(-1))
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
loss.item()

## 10. Autoregressive decoding

In [ ]:
def greedy_decode(model, source, source_mask, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id('[SOS]')
    eos_idx = tokenizer_tgt.token_to_id('[EOS]')
    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.full((1, 1), sos_idx, dtype=torch.long, device=device)
    while decoder_input.size(1) < max_len:
        decoder_mask = causal_mask(decoder_input.size(1)).to(device)
        decoder_output = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        probabilities = model.project(decoder_output[:, -1])
        next_token = probabilities.argmax(dim=-1).item()
        decoder_input = torch.cat([decoder_input, torch.tensor([[next_token]], device=device)], dim=1)
        if next_token == eos_idx:
            break
    return decoder_input.squeeze(0)

In [ ]:
model.eval()
sample = val_dataset[0]
source = sample['encoder_input'].unsqueeze(0).to(device)
source_mask = sample['encoder_mask'].to(device)
decoded = greedy_decode(model, source, source_mask, tokenizer_tgt, config['seq_len'], device)
prediction = tokenizer_tgt.decode(decoded.cpu().tolist(), skip_special_tokens=True)
sample['src_text'], sample['tgt_text'], prediction

## 11. Attention inspection

In [ ]:
model.eval()
with torch.no_grad():
    sample_batch = next(iter(val_loader))
    src = sample_batch['encoder_input'].to(device)
    src_mask = sample_batch['encoder_mask'].to(device)
    tgt = sample_batch['decoder_input'].to(device)
    tgt_mask = sample_batch['decoder_mask'].to(device)
    encoded = model.encode(src, src_mask)
    decoded = model.decode(encoded, src_mask, tgt, tgt_mask)

attention = model.encoder.layers[0].self_attention_block.attention_scores
attention.shape

## 12. Checkpointing

In [ ]:
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': config
}
torch.save(checkpoint, 'transformer_en_fr.pt')

## 13. Project training

The notebook contains the architecture and tensor-level implementation. The project training pipeline can be run from the repository root with `python train.py`.